# 09 — Resumo metodológico, ambiente e pacote

**Lê:** todos os JSON de decisão e o registro de acessos ao teste.  
**Grava:** `ambiente_execucao.json`, `requirements_execucao.txt` e o `.zip` da execução.

Os números do texto do TCC devem vir dos CSVs desta execução, nunca de outra versão.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "09_resumo_e_pacote"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
import json, shutil
from tcc_fluxo.io import sha256_arquivo, _ler_texto

sel = A.carregar_json("selecao_modelo")
exp1 = A.carregar_json("experimento1_modelo_escolhido") if A.existe("experimento1_modelo_escolhido") else None
div = A.carregar_json("divisao")
print("Modelo escolhido (seção 16):", sel["modelo_escolhido"], "| empate prático:", sel["modelos_em_empate_pratico"])
print("Experimento 1:", None if exp1 is None else exp1["decisao"])
print(pd.Series(div))

reg = A.dir / "registro_acesso_teste.jsonl"
acessos = pd.read_json(reg, lines=True) if reg.is_file() else pd.DataFrame()
print("Acessos ao teste registrados:"); display(acessos)

In [ ]:
inputs = {str(p.relative_to(C.RAIZ)): sha256_arquivo(p)
          for pasta in (C.caminho("dados_olist"), C.caminho("macro_cache")) for p in sorted(pasta.glob("*")) if p.is_file()}
amb = {"versoes": A.versions, "config": C.como_dict(), "input_sha256": inputs,
       "artefatos": {p.name: json.loads(_ler_texto(p))["sha256"] for p in sorted(A.dir.glob("*.meta.json"))}}
(A.saidas / "ambiente_execucao.json").write_text(json.dumps(amb, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
(A.saidas / "requirements_execucao.txt").write_text("\n".join(f"{k}=={v}" for k, v in A.versions.items() if k != "python" and v) + "\n", encoding="utf-8")
zip_base = C.caminho("artefatos").parent
arq = shutil.make_archive(str(zip_base), "zip", str(zip_base))
print("Pacote:", arq)

In [ ]:
A.finalizar()